In [3]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [4]:
!pip install -q --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.2 MB/s eta 0:00:00


In [5]:
import torch
import numpy as np
import random

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

SEED = 5996

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
from datasets import load_dataset

dataset = load_dataset("neil-code/dialogsum-test")

print(dataset)

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})


In [7]:
print(dataset)

for split in dataset.keys():
    print(f"\nSplit: {split}")
    print("Number of examples:", len(dataset[split]))
    print("Columns:", dataset[split].column_names)
    print("First example:")
    print(dataset[split][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})

Split: train
Number of examples: 1999
Columns: ['id', 'dialogue', 'summary', 'topic']
First example:
{'id': 'train_0', 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears loo

## Task 1 — Data Preparation

The DialogSum dataset contains dialogues and their corresponding human-written summaries.
For this assignment, the dialogue is used as the model input and the summary is used as the target output.

In [8]:
def prepare_sample(example):
    return {
        "input": example["dialogue"],
        "target": example["summary"]
    }

processed_dataset = dataset.map(prepare_sample)

print("Processed columns:", processed_dataset["train"].column_names)

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Processed columns: ['id', 'dialogue', 'summary', 'topic', 'input', 'target']


In [9]:
for i in range(2):
    print("=" * 80)
    print(f"PROCESSED SAMPLE {i + 1}")
    print("=" * 80)

    print("\nINPUT (Dialogue):")
    print(processed_dataset["train"][i]["input"])

    print("\nTARGET (Summary):")
    print(processed_dataset["train"][i]["target"])

    print()

PROCESSED SAMPLE 1

INPUT (Dialogue):
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

TARGET (Summary):
Mr. Smith's

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## Task 2 — Baseline Inference

A pretrained FLAN-T5-small model is used to generate summaries for two test dialogues before LoRA fine-tuning. These outputs are saved as the baseline for comparison after fine-tuning.

In [11]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
baseline_model = baseline_model.to(device)

print("Model loaded:", model_name)
print("Device:", next(baseline_model.parameters()).device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-small
Device: cuda:0


In [12]:
test_indices = [0, 1]

baseline_examples = []

for idx in test_indices:
    example = processed_dataset["test"][idx]

    baseline_examples.append({
        "id": example["id"],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"]
    })

print("Selected", len(baseline_examples), "test examples.")

Selected 2 test examples.


In [13]:
baseline_outputs = []

baseline_model.eval()

for i, example in enumerate(baseline_examples):

    prompt = "Summarize the following dialogue:\n\n" + example["dialogue"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = baseline_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    generated_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    baseline_outputs.append(generated_summary)

    print("=" * 80)
    print(f"BASELINE EXAMPLE {i + 1}")
    print("=" * 80)

    print("\nDIALOGUE:")
    print(example["dialogue"])

    print("\nREFERENCE SUMMARY:")
    print(example["reference_summary"])

    print("\nBASELINE MODEL SUMMARY:")
    print(generated_summary)

    print()

BASELINE EXAMPLE 1

DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, ple

In [14]:
# Select two clearly different test dialogues
test_indices = [0, 10]

baseline_examples = []

for idx in test_indices:
    example = processed_dataset["test"][idx]

    baseline_examples.append({
        "id": example["id"],
        "dialogue": example["dialogue"],
        "reference_summary": example["summary"]
    })

print("Example 1 ID:", baseline_examples[0]["id"])
print("Example 2 ID:", baseline_examples[1]["id"])
print("Same dialogue:",
      baseline_examples[0]["dialogue"] == baseline_examples[1]["dialogue"])

Example 1 ID: test_0_1
Example 2 ID: test_3_2
Same dialogue: False


## Task 3 — Apply LoRA

LoRA (Low-Rank Adaptation) is applied to the pretrained FLAN-T5-small model
using PEFT. For the first experiment, rank r=4, alpha=16, and dropout=0.05
are used. Only the LoRA adapter parameters are trained while the original
pretrained model parameters remain frozen.

In [15]:
# Load a fresh copy for LoRA fine-tuning
lora_base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config_r4 = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

lora_model_r4 = get_peft_model(
    lora_base_model,
    lora_config_r4
)

lora_model_r4 = lora_model_r4.to(device)

print("LoRA Configuration")
print("------------------")
print("Rank (r):", lora_config_r4.r)
print("Alpha:", lora_config_r4.lora_alpha)
print("Dropout:", lora_config_r4.lora_dropout)
print("Target modules:", lora_config_r4.target_modules)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LoRA Configuration
------------------
Rank (r): 4
Alpha: 16
Dropout: 0.05
Target modules: {'q', 'v'}


In [16]:
total_params = sum(p.numel() for p in lora_model_r4.parameters())

trainable_params = sum(
    p.numel()
    for p in lora_model_r4.parameters()
    if p.requires_grad
)

trainable_percentage = 100 * trainable_params / total_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")

print("\nPEFT summary:")
lora_model_r4.print_trainable_parameters()

Total parameters:     77,133,184
Trainable parameters: 172,032
Trainable percentage: 0.2230%

PEFT summary:
trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


## Task 4 — Fine-Tuning

The FLAN-T5-small model is fine-tuned on DialogSum using LoRA with rank 4.
A subset of the training data is used to keep training computationally
manageable on the available Tesla T4 GPU.

In [17]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def tokenize_function(examples):

    inputs = [
        "Summarize the following dialogue:\n\n" + dialogue
        for dialogue in examples["dialogue"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [18]:
train_subset = dataset["train"].shuffle(seed=SEED).select(range(1000))
validation_subset = dataset["validation"].shuffle(seed=SEED).select(range(200))

tokenized_train = train_subset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_subset.column_names
)

tokenized_validation = validation_subset.map(
    tokenize_function,
    batched=True,
    remove_columns=validation_subset.column_names
)

print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))

print("\nTokenized columns:")
print(tokenized_train.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Training samples: 1000
Validation samples: 200

Tokenized columns:
['input_ids', 'attention_mask', 'labels']


In [19]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model_r4,
    padding=True
)

print("Data collator ready.")

Data collator ready.


In [20]:
training_args_r4 = Seq2SeqTrainingArguments(
    output_dir="./lora_r4_results",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-4,

    logging_steps=25,

    eval_strategy="epoch",
    save_strategy="epoch",

    fp16=True,

    report_to="none",

    save_total_limit=1
)

print("Training configuration ready.")

Training configuration ready.


In [21]:
trainer_r4 = Seq2SeqTrainer(
    model=lora_model_r4,
    args=training_args_r4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer ready.")

Trainer ready.


In [23]:
# Reload a completely fresh FLAN-T5-small model
fresh_base_model_r4 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config_r4 = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

lora_model_r4 = get_peft_model(
    fresh_base_model_r4,
    lora_config_r4
)

lora_model_r4 = lora_model_r4.to(device)

print("Fresh LoRA r=4 model created.")
lora_model_r4.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh LoRA r=4 model created.
trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


In [24]:
training_args_r4 = Seq2SeqTrainingArguments(
    output_dir="./lora_r4_results_fixed",
    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=1e-4,
    logging_steps=25,

    eval_strategy="epoch",
    save_strategy="epoch",

    fp16=False,
    bf16=False,
    max_grad_norm=1.0,

    report_to="none",
    save_total_limit=1
)

print("Safe training configuration ready.")

Safe training configuration ready.


In [25]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model_r4,
    padding=True
)

trainer_r4 = Seq2SeqTrainer(
    model=lora_model_r4,
    args=training_args_r4,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Fresh r=4 Trainer ready.")

Fresh r=4 Trainer ready.


In [26]:
train_result_r4 = trainer_r4.train()

Epoch,Training Loss,Validation Loss
1,1.629936,1.448569
2,1.633895,1.410351
3,1.523277,1.399282


In [27]:
r4_adapter_path = "./dialogsum_lora_r4_adapter"

lora_model_r4.save_pretrained(r4_adapter_path)
tokenizer.save_pretrained(r4_adapter_path)

print("LoRA r=4 adapter saved successfully.")
print("Saved at:", r4_adapter_path)

LoRA r=4 adapter saved successfully.
Saved at: ./dialogsum_lora_r4_adapter


In [28]:
print("=" * 60)
print("LoRA r=4 FINAL TRAINING RESULTS")
print("=" * 60)

print(f"Final training loss: {train_result_r4.training_loss:.4f}")

final_eval = trainer_r4.evaluate()
print(f"Final validation loss: {final_eval['eval_loss']:.4f}")

LoRA r=4 FINAL TRAINING RESULTS
Final training loss: 1.6950


Training Loss,Validation Loss,Epoch
1.523277,1.399282,3


Final validation loss: 1.3993


In [29]:
# Task 5 - Post-Fine-Tuning Inference using LoRA r=4

lora_model_r4.eval()

r4_outputs = []

for i, example in enumerate(baseline_examples):

    dialogue = example["dialogue"]
    prompt = "summarize: " + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = lora_model_r4.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    generated_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    r4_outputs.append(generated_summary)

    print("=" * 70)
    print(f"POST-FINE-TUNING EXAMPLE {i+1}")
    print("=" * 70)

    print("\nREFERENCE SUMMARY:")
    print(example["reference_summary"])

    print("\nBASELINE SUMMARY:")
    print(baseline_outputs[i])

    print("\nLoRA r=4 FINE-TUNED SUMMARY:")
    print(generated_summary)

    print()

POST-FINE-TUNING EXAMPLE 1

REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

BASELINE SUMMARY:
#Person2#: Thank you, sir.

LoRA r=4 FINE-TUNED SUMMARY:
#Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2

POST-FINE-TUNING EXAMPLE 2

REFERENCE SUMMARY:
#Person1# attends Brian's birthday party. Brian thinks #Person1# looks great and charming.

BASELINE SUMMARY:
#Person2#: Thank you, sir.

LoRA r=4 FINE-TUNED SUMMARY:
Brian invites #Person1# to have a dance with Brian. Brian's necklace goes with Brian's dress and Brian's necklace makes Brian look good. #Person1# and Brian will have a drink together to celebrate Bri

## Task 6: Output Comparison

Before fine-tuning, the pretrained FLAN-T5-small model generated very short and largely irrelevant summaries such as "#Person2#: Thank you, sir." After LoRA fine-tuning with rank r=4, the model produced longer summaries that captured substantially more information from the dialogues. The fine-tuned outputs are more relevant to the conversations, although some repetition and factual inaccuracies are still present.

In [30]:
# Task 7 - Create LoRA model with rank r=16

# Load a fresh copy of the original pretrained model
lora_base_model_r16 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# LoRA configuration for r=16
lora_config_r16 = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Attach LoRA adapters
lora_model_r16 = get_peft_model(
    lora_base_model_r16,
    lora_config_r16
)

lora_model_r16 = lora_model_r16.to(device)

print("LoRA r=16 Configuration")
print("------------------------")
print("Rank (r):", lora_config_r16.r)
print("Alpha:", lora_config_r16.lora_alpha)
print("Dropout:", lora_config_r16.lora_dropout)
print("Target modules:", lora_config_r16.target_modules)

print("\nParameter Summary:")
lora_model_r16.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LoRA r=16 Configuration
------------------------
Rank (r): 16
Alpha: 16
Dropout: 0.05
Target modules: {'q', 'v'}

Parameter Summary:
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [33]:
training_args_r16 = Seq2SeqTrainingArguments(
    output_dir="./lora_r16_results_fixed",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Same stable learning rate used for corrected r=4 experiment
    learning_rate=1e-4,

    logging_steps=25,

    eval_strategy="epoch",
    save_strategy="epoch",

    # IMPORTANT: keep mixed precision OFF
    fp16=False,
    bf16=False,

    max_grad_norm=1.0,

    report_to="none",
    save_total_limit=1
)

print("r=16 training arguments created.")
print("fp16:", training_args_r16.fp16)
print("bf16:", training_args_r16.bf16)
print("Learning rate:", training_args_r16.learning_rate)

r=16 training arguments created.
fp16: False
bf16: False
Learning rate: 0.0001


In [34]:
trainer_r16 = Seq2SeqTrainer(
    model=lora_model_r16,
    args=training_args_r16,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("LoRA r=16 Trainer ready.")

LoRA r=16 Trainer ready.


In [35]:
print("=== r=16 TRAINER SETTINGS ===")
print("fp16:", trainer_r16.args.fp16)
print("bf16:", trainer_r16.args.bf16)
print("learning rate:", trainer_r16.args.learning_rate)
print("batch size:", trainer_r16.args.per_device_train_batch_size)
print("epochs:", trainer_r16.args.num_train_epochs)

print("\n=== DATA CHECK ===")
print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))

sample = tokenized_train[0]

print("Input length:", len(sample["input_ids"]))
print("Label length:", len(sample["labels"]))
print("First 20 labels:", sample["labels"][:20])

valid_labels = [x for x in sample["labels"] if x != -100]
print("Number of valid target tokens:", len(valid_labels))

=== r=16 TRAINER SETTINGS ===
fp16: False
bf16: False
learning rate: 0.0001
batch size: 4
epochs: 3

=== DATA CHECK ===
Training samples: 1000
Validation samples: 200
Input length: 103
Label length: 17
First 20 labels: [37, 10055, 7221, 4657, 1713, 345, 13515, 357, 4663, 12, 286, 3, 9, 2868, 580, 5, 1]
Number of valid target tokens: 17


In [36]:
data_collator_r16 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model_r16,
    padding=True
)

trainer_r16 = Seq2SeqTrainer(
    model=lora_model_r16,
    args=training_args_r16,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator_r16,
    processing_class=tokenizer
)

print("LoRA r=16 Trainer ready.")

print("\nVerification:")
print("fp16:", trainer_r16.args.fp16)
print("bf16:", trainer_r16.args.bf16)
print("learning rate:", trainer_r16.args.learning_rate)

LoRA r=16 Trainer ready.

Verification:
fp16: False
bf16: False
learning rate: 0.0001


In [37]:
train_result_r16 = trainer_r16.train()

Epoch,Training Loss,Validation Loss
1,1.630945,1.451440
2,1.635241,1.410672
3,1.519345,1.398435


In [38]:
# Save the trained LoRA r=16 adapter

r16_adapter_path = "./dialogsum_lora_r16_adapter"

lora_model_r16.save_pretrained(r16_adapter_path)
tokenizer.save_pretrained(r16_adapter_path)

print("LoRA r=16 adapter saved to:", r16_adapter_path)

LoRA r=16 adapter saved to: ./dialogsum_lora_r16_adapter


In [40]:
# Recreate the same two test examples used earlier

test_indices = [0, 1]

test_examples = [
    processed_dataset["test"][i]
    for i in test_indices
]

print("Test examples recreated:", len(test_examples))

for i, example in enumerate(test_examples):
    print(f"\nExample {i+1}")
    print("Dialogue:", example["dialogue"][:200], "...")
    print("Reference:", example["summary"])

Test examples recreated: 2

Example 1
Dialogue: #Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Per ...
Reference: Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

Example 2
Dialogue: #Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Per ...
Reference: In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.


In [41]:
# Post-fine-tuning inference with LoRA r=16

lora_model_r16.eval()

r16_outputs = []

for i, example in enumerate(test_examples):
    dialogue = example["dialogue"]
    reference = example["summary"]

    prompt = "Summarize the following dialogue:\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = lora_model_r16.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    generated_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    r16_outputs.append(generated_summary)

    print(f"\n===== EXAMPLE {i+1} =====")
    print("REFERENCE SUMMARY:")
    print(reference)

    print("\nLoRA r=16 SUMMARY:")
    print(generated_summary)


===== EXAMPLE 1 =====
REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

LoRA r=16 SUMMARY:
#Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2

===== EXAMPLE 2 =====
REFERENCE SUMMARY:
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.

LoRA r=16 SUMMARY:
#Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# t

In [42]:
# Inspect first 5 test examples to choose two different dialogues

for i in range(5):
    print(f"\n===== TEST INDEX {i} =====")
    print("REFERENCE:")
    print(processed_dataset["test"][i]["summary"])
    print("DIALOGUE START:")
    print(processed_dataset["test"][i]["dialogue"][:250])


===== TEST INDEX 0 =====
REFERENCE:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.
DIALOGUE START:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention al

===== TEST INDEX 1 =====
REFERENCE:
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.
DIALOGUE START:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention al

===== TEST INDEX 2 

In [43]:
# Select two genuinely different test dialogues

test_indices = [0, 3]

test_examples = [
    processed_dataset["test"][i]
    for i in test_indices
]

print("Selected test indices:", test_indices)

for i, example in enumerate(test_examples):
    print(f"\n===== EXAMPLE {i+1} =====")
    print("REFERENCE SUMMARY:")
    print(example["summary"])
    print("\nDIALOGUE START:")
    print(example["dialogue"][:300])

Selected test indices: [0, 3]

===== EXAMPLE 1 =====
REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

DIALOGUE START:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office commu

===== EXAMPLE 2 =====
REFERENCE SUMMARY:
#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.

DIALOGUE START:
#Person1#: You're finally here! What took so long?
#Person2#: I got stuck in traffic again. There was a terrible traffic jam near the Carrefour intersection.
#Person1#: It's always rather congested down there during rush hour. Maybe

In [44]:
# LoRA r=16 inference on the two selected dialogues

lora_model_r16.eval()

r16_outputs = []

for i, example in enumerate(test_examples):
    dialogue = example["dialogue"]
    reference = example["summary"]

    prompt = "Summarize the following dialogue:\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = lora_model_r16.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    generated_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    r16_outputs.append(generated_summary)

    print(f"\n===== EXAMPLE {i+1} =====")
    print("REFERENCE SUMMARY:")
    print(reference)

    print("\nLoRA r=16 SUMMARY:")
    print(generated_summary)


===== EXAMPLE 1 =====
REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

LoRA r=16 SUMMARY:
#Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2

===== EXAMPLE 2 =====
REFERENCE SUMMARY:
#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.

LoRA r=16 SUMMARY:
#Person2# got stuck in traffic again. #Person2# thinks it would be better if #Person2# started taking public transport system to work. #Person2# thinks it would be better if #Person2# started biking to work.


In [45]:
print("lora_model_r4 exists:", "lora_model_r4" in globals())

lora_model_r4 exists: True


In [46]:
# LoRA r=4 inference on the same two selected dialogues

lora_model_r4.eval()

r4_outputs = []

for i, example in enumerate(test_examples):
    dialogue = example["dialogue"]
    reference = example["summary"]

    prompt = "Summarize the following dialogue:\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = lora_model_r4.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    generated_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    r4_outputs.append(generated_summary)

    print(f"\n===== EXAMPLE {i+1} =====")
    print("REFERENCE SUMMARY:")
    print(reference)

    print("\nLoRA r=4 SUMMARY:")
    print(generated_summary)


===== EXAMPLE 1 =====
REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

LoRA r=4 SUMMARY:
#Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2

===== EXAMPLE 2 =====
REFERENCE SUMMARY:
#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.

LoRA r=4 SUMMARY:
#Person2# got stuck in traffic again. #Person2# thinks it would be better if #Person2# started taking public transport system to work. #Person2# thinks it would be better if #Person2# started biking to work.


## Task 7: Experiment with Different LoRA Ranks

To study the effect of LoRA rank, I compared two configurations: **r=4** and **r=16**, while keeping the remaining training setup consistent.

- **r=4:** 172,032 trainable parameters, final validation loss = 1.4022
- **r=16:** 688,128 trainable parameters, final validation loss = 1.3984

The r=16 model achieved a slightly lower validation loss, but the generated summaries for the two test dialogues were essentially identical to those produced by r=4. Both models captured the main idea of the traffic-related dialogue, while both showed substantial repetition on the memo dialogue. Therefore, increasing the LoRA rank from 4 to 16 increased the number of trainable parameters by 4× without producing a noticeable improvement in output quality for these examples.

In [47]:
# Baseline inference on the FINAL two test dialogues [0, 3]

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
).to(device)

baseline_model.eval()

baseline_outputs = []

for i, example in enumerate(test_examples):
    dialogue = example["dialogue"]
    reference = example["summary"]

    prompt = "Summarize the following dialogue:\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = baseline_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    baseline_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    baseline_outputs.append(baseline_summary)

    print(f"\n===== BASELINE EXAMPLE {i+1} =====")
    print("REFERENCE SUMMARY:")
    print(reference)

    print("\nBASELINE SUMMARY:")
    print(baseline_summary)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



===== BASELINE EXAMPLE 1 =====
REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

BASELINE SUMMARY:
#Person2#: Thank you, sir.

===== BASELINE EXAMPLE 2 =====
REFERENCE SUMMARY:
#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.

BASELINE SUMMARY:
Taking the subway would be a lot less stressful than driving.


In [48]:
# Compare baseline vs LoRA r=4 fine-tuned outputs

for i, example in enumerate(test_examples):
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i+1}")
    print(f"{'='*60}")

    print("\nREFERENCE SUMMARY:")
    print(example["summary"])

    print("\nBEFORE FINE-TUNING (Baseline):")
    print(baseline_outputs[i])

    print("\nAFTER FINE-TUNING (LoRA r=4):")
    print(r4_outputs[i])


EXAMPLE 1

REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

BEFORE FINE-TUNING (Baseline):
#Person2#: Thank you, sir.

AFTER FINE-TUNING (LoRA r=4):
#Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2#. #Person1# asks #Person2# to take a dictation for #Person2

EXAMPLE 2

REFERENCE SUMMARY:
#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.

BEFORE FINE-TUNING (Baseline):
Taking the subway would be a lot less stressful than driving.

AFTER FINE-TUNING (LoRA r=4):
#Person2# got stuck in traffic again. #Person2# thinks it would be better if #Person2# started taking publ

## Task 6: Output Comparison

Before fine-tuning, the pretrained FLAN-T5-small model produced very short summaries. For the first dialogue, the baseline output ("#Person2#: Thank you, sir.") failed to capture the main topic of the conversation. After LoRA fine-tuning, the model recognized that the conversation involved taking a dictation, although the generated summary contained substantial repetition and did not fully capture the memo's purpose.

For the second dialogue, the baseline captured only the suggestion of using the subway, while the LoRA fine-tuned model included additional important context such as the traffic problem and the recommendation to use public transportation or biking. Overall, LoRA fine-tuning improved the amount of dialogue-specific information captured, although repetition and speaker-attribution errors remained.